# Project: Arabic Bank Check Amount Extraction and Processing


## Google Colab Setup


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import ast
import glob
import os
import random
import re
import shutil
import subprocess

subprocess.run(["pip", "install", "-q", "ultralytics", "editdistance"], check=True)

DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_FOLDER_NAME = "Arabic-Bank-Check-Amount-Extraction-and-Processing"
PROJECT_DIR = DRIVE_ROOT / PROJECT_FOLDER_NAME

if not PROJECT_DIR.exists():
    matches = sorted(DRIVE_ROOT.glob("Arabic-Bank-Check-*"))
    if len(matches) == 1:
        PROJECT_DIR = matches[0]
    else:
        raise FileNotFoundError(
            f"Could not find {PROJECT_FOLDER_NAME!r} under {DRIVE_ROOT}. "
            "Update PROJECT_FOLDER_NAME to match the Google Drive folder name."
        )

IMAGES_DIR = PROJECT_DIR / "Images"
BOUNDING_BOXES_DIR = PROJECT_DIR / "BoundingBoxes"
COURTESY_AMOUNTS_DIR = PROJECT_DIR / "CourtesyAmounts"
COURTESY_AMOUNTS_RAW_DIR = PROJECT_DIR / "CourtesyAmounts_raw"
LEGAL_AMOUNTS_RAW_TEXT_DIR = PROJECT_DIR / "LegalAmounts_raw_text"
LEGAL_AMOUNTS_TOKENIZED_DIR = PROJECT_DIR / "LegalAmounts_tokenized"

WORK_DIR = Path("/content/arabic_bank_check_work")
YOLO_DATASET_DIR = WORK_DIR / "yolo_dataset"
COURTESY_IMAGES_DIR = WORK_DIR / "courtesy_images"
LEGAL_IMAGES_DIR = WORK_DIR / "legal_images"

OUTPUTS_DIR = PROJECT_DIR / "outputs"
RUNS_DIR = PROJECT_DIR / "runs"
PART_A_OUTPUT = OUTPUTS_DIR / "partA_output.txt"
PART_A_METRICS = OUTPUTS_DIR / "partA_metrics.txt"
PART_A_AUDIT = OUTPUTS_DIR / "partA_annotation_audit.json"
PART_A_SPLIT = OUTPUTS_DIR / "partA_split.json"
PART_B_OUTPUT = OUTPUTS_DIR / "partB_output.txt"
PART_C_OUTPUT = OUTPUTS_DIR / "partC_output.txt"

for directory in [WORK_DIR, YOLO_DATASET_DIR, COURTESY_IMAGES_DIR, LEGAL_IMAGES_DIR, OUTPUTS_DIR, RUNS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

required_dirs = {
    "Images": IMAGES_DIR,
    "BoundingBoxes": BOUNDING_BOXES_DIR,
    "CourtesyAmounts": COURTESY_AMOUNTS_DIR,
    "CourtesyAmounts_raw": COURTESY_AMOUNTS_RAW_DIR,
    "LegalAmounts_raw_text": LEGAL_AMOUNTS_RAW_TEXT_DIR,
    "LegalAmounts_tokenized": LEGAL_AMOUNTS_TOKENIZED_DIR,
}

missing = [name for name, folder in required_dirs.items() if not folder.exists()]
if missing:
    raise FileNotFoundError(f"Missing required project folders in {PROJECT_DIR}: {missing}")

def normalize_check_id(filename, prefix=None):
    name = Path(str(filename).lstrip("\ufeff")).stem
    if prefix and name.startswith(prefix):
        name = name[len(prefix):]
    if name.startswith("ac") and name[2:].isdigit():
        return "ac" + name[2:].zfill(5)
    match = re.search(r"(\d+)", name)
    if match:
        return "ac" + match.group(1).zfill(5)
    return name

print("Project directory:", PROJECT_DIR)
for name, folder in required_dirs.items():
    count = len(list(folder.glob("*.txt"))) if name != "Images" else len(list(folder.glob("*.tif")))
    print(f"{name}: {count} files")
print("Outputs:", OUTPUTS_DIR)


## Prepare YOLO Dataset

In [ ]:
import json
import math
import random
import shutil
from collections import Counter

IMAGE_EXTENSIONS = {".tif", ".tiff", ".jpg", ".jpeg", ".png"}
YOLO_CLASSES = {0: "legal_amount", 1: "courtesy_amount"}
MIN_BOX_AREA_FOR_REVIEW = 0.001

IMAGES_DIR = Path(IMAGES_DIR)
LABELS_DIR = Path(BOUNDING_BOXES_DIR)
OUTPUT_DIR = str(YOLO_DATASET_DIR)

for split in ["train", "val"]:
    image_split_dir = YOLO_DATASET_DIR / "images" / split
    label_split_dir = YOLO_DATASET_DIR / "labels" / split
    if image_split_dir.exists():
        shutil.rmtree(image_split_dir)
    if label_split_dir.exists():
        shutil.rmtree(label_split_dir)
    image_split_dir.mkdir(parents=True, exist_ok=True)
    label_split_dir.mkdir(parents=True, exist_ok=True)

image_map = {}
duplicate_image_ids = []
for image_path in sorted(IMAGES_DIR.iterdir()):
    if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
        continue
    check_id = normalize_check_id(image_path.name)
    if check_id in image_map:
        duplicate_image_ids.append(check_id)
    image_map[check_id] = image_path

def yolo_to_edges(x_center, y_center, width, height):
    return (
        x_center - width / 2,
        y_center - height / 2,
        x_center + width / 2,
        y_center + height / 2,
    )

def edges_to_yolo(x1, y1, x2, y2):
    width = x2 - x1
    height = y2 - y1
    return ((x1 + x2) / 2, (y1 + y2) / 2, width, height)

def parse_and_clean_label(label_path):
    boxes = []
    warnings = []
    lines = [line.strip() for line in label_path.read_text(encoding="utf-8-sig").splitlines() if line.strip()]

    for line_number, line in enumerate(lines, start=1):
        parts = line.split()
        if len(parts) != 5:
            raise ValueError(f"line {line_number}: expected 5 YOLO fields, found {len(parts)}")

        try:
            class_id = int(parts[0])
            x_center, y_center, width, height = map(float, parts[1:])
        except ValueError as exc:
            raise ValueError(f"line {line_number}: non-numeric YOLO field") from exc

        if class_id not in YOLO_CLASSES:
            raise ValueError(f"line {line_number}: class {class_id} is not one of {sorted(YOLO_CLASSES)}")
        if not all(math.isfinite(value) for value in [x_center, y_center, width, height]):
            raise ValueError(f"line {line_number}: non-finite coordinate")
        if width <= 0 or height <= 0:
            raise ValueError(f"line {line_number}: non-positive box size")
        if not all(0 <= value <= 1 for value in [x_center, y_center, width, height]):
            raise ValueError(f"line {line_number}: YOLO coordinate outside [0, 1]")

        x1, y1, x2, y2 = yolo_to_edges(x_center, y_center, width, height)
        clipped_edges = (max(0.0, x1), max(0.0, y1), min(1.0, x2), min(1.0, y2))
        was_clipped = clipped_edges != (x1, y1, x2, y2)
        x1, y1, x2, y2 = clipped_edges
        if x2 <= x1 or y2 <= y1:
            raise ValueError(f"line {line_number}: box disappears after clipping")

        x_center, y_center, width, height = edges_to_yolo(x1, y1, x2, y2)
        area = width * height
        if was_clipped:
            warnings.append({"file": label_path.name, "line": line_number, "reason": "clipped_to_image_bounds"})
        if area < MIN_BOX_AREA_FOR_REVIEW:
            warnings.append({"file": label_path.name, "line": line_number, "reason": "very_small_box", "area": area})

        boxes.append((class_id, x_center, y_center, width, height))

    class_counts = Counter(class_id for class_id, *_ in boxes)
    if len(boxes) != 2 or class_counts.get(0, 0) != 1 or class_counts.get(1, 0) != 1:
        raise ValueError(f"expected exactly one legal box and one courtesy box, found {dict(class_counts)}")

    return sorted(boxes, key=lambda item: item[0]), warnings

valid_records = []
audit = {
    "duplicate_image_ids": duplicate_image_ids,
    "missing_images_for_labels": [],
    "invalid_label_files": [],
    "warnings": [],
}

for label_path in sorted(LABELS_DIR.glob("*.txt")):
    check_id = normalize_check_id(label_path.name)
    image_path = image_map.get(check_id)
    if image_path is None:
        audit["missing_images_for_labels"].append(label_path.name)
        continue

    try:
        cleaned_boxes, warnings = parse_and_clean_label(label_path)
    except ValueError as exc:
        audit["invalid_label_files"].append({"file": label_path.name, "error": str(exc)})
        continue

    audit["warnings"].extend(warnings)
    valid_records.append({
        "check_id": check_id,
        "image_path": image_path,
        "label_path": label_path,
        "boxes": cleaned_boxes,
    })

if audit["duplicate_image_ids"] or audit["invalid_label_files"]:
    raise ValueError(f"Part-A annotation audit failed. See details: {audit}")
if not valid_records:
    raise ValueError("No valid image-label pairs were found. Check Images/ and BoundingBoxes/ filenames.")

valid_ids = {record["check_id"] for record in valid_records}
audit["images_without_labels"] = sorted(set(image_map) - valid_ids)
audit["valid_image_label_pairs"] = len(valid_records)
audit["total_images"] = len(image_map)
audit["total_label_files"] = len(list(LABELS_DIR.glob("*.txt")))

rng = random.Random(42)
rng.shuffle(valid_records)
split_idx = int(len(valid_records) * 0.85)
train_records = valid_records[:split_idx]
val_records = valid_records[split_idx:]

def write_clean_label(label_path, boxes):
    lines = [f"{class_id} {x:.8f} {y:.8f} {w:.8f} {h:.8f}" for class_id, x, y, w, h in boxes]
    label_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

def copy_records(records, split_name):
    for record in records:
        image_path = record["image_path"]
        target_image = YOLO_DATASET_DIR / "images" / split_name / image_path.name
        target_label = YOLO_DATASET_DIR / "labels" / split_name / f"{image_path.stem}.txt"
        shutil.copy2(image_path, target_image)
        write_clean_label(target_label, record["boxes"])

copy_records(train_records, "train")
copy_records(val_records, "val")

split_summary = {
    "train": [record["check_id"] for record in train_records],
    "val": [record["check_id"] for record in val_records],
    "seed": 42,
    "train_fraction": 0.85,
}
PART_A_AUDIT.write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8")
PART_A_SPLIT.write_text(json.dumps(split_summary, ensure_ascii=False, indent=2), encoding="utf-8")

print("Part-A annotation audit")
print(f"  Images found:              {audit['total_images']}")
print(f"  Label files found:         {audit['total_label_files']}")
print(f"  Valid image-label pairs:   {audit['valid_image_label_pairs']}")
print(f"  Images without labels:     {len(audit['images_without_labels'])}")
print(f"  Non-fatal label warnings:  {len(audit['warnings'])}")
print(f"  Train set:                 {len(train_records)}")
print(f"  Validation set:            {len(val_records)}")
print("YOLO dataset prepared at:", YOLO_DATASET_DIR)
print("Audit saved to:", PART_A_AUDIT)
print("Split saved to:", PART_A_SPLIT)


## Convert TIF to JPG

In [ ]:
from PIL import Image


def convert_split_to_jpg(split_name):
    image_dir = YOLO_DATASET_DIR / "images" / split_name
    tif_paths = sorted(list(image_dir.glob("*.tif")) + list(image_dir.glob("*.tiff")))
    converted = 0

    for tif_path in tif_paths:
        jpg_path = tif_path.with_suffix(".jpg")
        with Image.open(tif_path) as image:
            image.convert("RGB").save(jpg_path, "JPEG", quality=95)
        tif_path.unlink()
        converted += 1

    jpg_count = len(list(image_dir.glob("*.jpg")))
    print(f"{split_name}: converted {converted} TIFF files; {jpg_count} JPG files ready")

print("Converting YOLO images to JPG")
convert_split_to_jpg("train")
convert_split_to_jpg("val")


## Train YOLO Model

In [ ]:
import yaml
from ultralytics import YOLO

data_yaml = {
    "path": str(YOLO_DATASET_DIR),
    "train": "images/train",
    "val": "images/val",
    "nc": 2,
    "names": {0: "legal_amount", 1: "courtesy_amount"},
}

YAML_PATH = YOLO_DATASET_DIR / "data.yaml"
YAML_PATH.write_text(yaml.dump(data_yaml, default_flow_style=False), encoding="utf-8")

print("data.yaml created at:", YAML_PATH)
print(yaml.dump(data_yaml, default_flow_style=False))

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(YAML_PATH),
    epochs=100,
    batch=16,
    imgsz=640,
    optimizer="AdamW",
    lr0=0.001,
    momentum=0.9,
    weight_decay=0.0005,
    amp=True,
    patience=100,
    project=str(RUNS_DIR),
    name="partA_yolov8n",
    exist_ok=True,
    save=True,
    verbose=True,
    deterministic=False,
)

print("Training complete")


## Part A — Generate Predictions

In [ ]:
import glob
from PIL import Image
from ultralytics import YOLO


def latest_best_model():
    candidates = []
    candidates.extend(glob.glob(str(RUNS_DIR / "**" / "weights" / "best.pt"), recursive=True))
    candidates.extend(glob.glob("/content/runs/detect/train*/weights/best.pt"))
    candidates.extend(glob.glob("/content/runs/train*/weights/best.pt"))
    if not candidates:
        raise FileNotFoundError("No best.pt file was found. Run the Part-A YOLO training cell first.")
    return max(candidates, key=os.path.getmtime)


def clip_xyxy(box, width, height):
    x1, y1, x2, y2 = [int(round(value)) for value in box]
    x1 = max(0, min(width, x1))
    y1 = max(0, min(height, y1))
    x2 = max(0, min(width, x2))
    y2 = max(0, min(height, y2))
    if x2 <= x1 or y2 <= y1:
        return [0, 0, 0, 0]
    return [x1, y1, x2, y2]

BEST_MODEL_PATH = latest_best_model()
model = YOLO(BEST_MODEL_PATH)
print("Using model:", BEST_MODEL_PATH)

VAL_IMAGES_DIR = YOLO_DATASET_DIR / "images" / "val"
val_images = sorted(VAL_IMAGES_DIR.glob("*.jpg"))
if not val_images:
    raise ValueError(f"No validation JPG images found in {VAL_IMAGES_DIR}")

output_lines = []
missed = {"courtesy_amount": 0, "legal_amount": 0}

for image_path in val_images:
    with Image.open(image_path) as image:
        image_width, image_height = image.size

    result = model.predict(str(image_path), imgsz=640, conf=0.25, verbose=False)[0]
    best_by_class = {
        0: {"confidence": -1.0, "box": [0, 0, 0, 0]},
        1: {"confidence": -1.0, "box": [0, 0, 0, 0]},
    }

    for box in result.boxes:
        class_id = int(box.cls[0].item())
        if class_id not in best_by_class:
            continue
        confidence = float(box.conf[0].item())
        if confidence <= best_by_class[class_id]["confidence"]:
            continue
        best_by_class[class_id] = {
            "confidence": confidence,
            "box": clip_xyxy(box.xyxy[0].tolist(), image_width, image_height),
        }

    legal_box = best_by_class[0]["box"]
    courtesy_box = best_by_class[1]["box"]
    if courtesy_box == [0, 0, 0, 0]:
        missed["courtesy_amount"] += 1
    if legal_box == [0, 0, 0, 0]:
        missed["legal_amount"] += 1

    output_name = image_path.with_suffix(".tif").name
    output_lines.append(
        f"{output_name} "
        f"{courtesy_box[0]} {courtesy_box[1]} {courtesy_box[2]} {courtesy_box[3]} "
        f"{legal_box[0]} {legal_box[1]} {legal_box[2]} {legal_box[3]}"
    )

PART_A_OUTPUT.write_text("\n".join(output_lines) + "\n", encoding="utf-8")

print("PART A OUTPUT SUMMARY")
print(f"  Validation images processed: {len(output_lines)}")
print(f"  Missed courtesy fields:      {missed['courtesy_amount']}")
print(f"  Missed legal fields:         {missed['legal_amount']}")
print(f"  Output saved to:             {PART_A_OUTPUT}")
print("Sample output:")
for line in output_lines[:5]:
    print(" ", line)


## Helper Functions for Evaluation

In [ ]:
import numpy as np
from PIL import Image

IOU_THRESHOLDS = (0.50, 0.75, 0.90)
CLASS_NAMES = {0: "legal_amount", 1: "courtesy_amount"}


def calculate_iou(box1, box2):
    x_left = max(box1[0], box2[0])
    y_top = max(box1[1], box2[1])
    x_right = min(box1[2], box2[2])
    y_bottom = min(box1[3], box2[3])

    intersection = max(0, x_right - x_left) * max(0, y_bottom - y_top)
    area1 = max(0, box1[2] - box1[0]) * max(0, box1[3] - box1[1])
    area2 = max(0, box2[2] - box2[0]) * max(0, box2[3] - box2[1])
    union = area1 + area2 - intersection
    return 0.0 if union <= 0 else intersection / union


def convert_yolo_to_abs_coords(x_center, y_center, width, height, img_width, img_height):
    x1 = max(0, int(round((x_center - width / 2) * img_width)))
    y1 = max(0, int(round((y_center - height / 2) * img_height)))
    x2 = min(img_width, int(round((x_center + width / 2) * img_width)))
    y2 = min(img_height, int(round((y_center + height / 2) * img_height)))
    return [x1, y1, x2, y2]


def summarize_iou_scores(iou_scores, thresholds=IOU_THRESHOLDS):
    total = len(iou_scores)
    mean_iou = float(np.mean(iou_scores)) if iou_scores else 0.0
    accuracies = {threshold: 0.0 for threshold in thresholds}
    if total:
        accuracies = {threshold: sum(iou >= threshold for iou in iou_scores) / total for threshold in thresholds}
    return {"total": total, "mean_iou": mean_iou, "accuracies": accuracies}


## Load Ground Truth and Predictions

In [ ]:
VAL_IMAGES_DIR = YOLO_DATASET_DIR / "images" / "val"
VAL_LABELS_DIR = YOLO_DATASET_DIR / "labels" / "val"
val_images = sorted(VAL_IMAGES_DIR.glob("*.jpg"))

if not val_images:
    raise ValueError(f"No validation images found in {VAL_IMAGES_DIR}")
if not PART_A_OUTPUT.exists():
    raise FileNotFoundError(f"Part-A prediction file not found: {PART_A_OUTPUT}")

image_dimensions = {}
ground_truths = {}

for image_path in val_images:
    with Image.open(image_path) as image:
        image_dimensions[image_path.stem] = image.size

    label_path = VAL_LABELS_DIR / f"{image_path.stem}.txt"
    if not label_path.exists():
        raise FileNotFoundError(f"Missing validation label file: {label_path}")

    boxes = {}
    image_width, image_height = image_dimensions[image_path.stem]
    for line_number, line in enumerate(label_path.read_text(encoding="utf-8").splitlines(), start=1):
        if not line.strip():
            continue
        parts = line.split()
        if len(parts) != 5:
            raise ValueError(f"Malformed label line in {label_path}, line {line_number}: {line}")
        class_id = int(parts[0])
        x_center, y_center, width, height = map(float, parts[1:])
        boxes[class_id] = convert_yolo_to_abs_coords(x_center, y_center, width, height, image_width, image_height)

    if set(boxes) != {0, 1}:
        raise ValueError(f"Expected one legal and one courtesy box in {label_path}, found classes {sorted(boxes)}")
    ground_truths[image_path.stem] = boxes

predictions = {}
for line_number, line in enumerate(PART_A_OUTPUT.read_text(encoding="utf-8").splitlines(), start=1):
    if not line.strip():
        continue
    parts = line.split()
    if len(parts) != 9:
        raise ValueError(f"Malformed Part-A output line {line_number}: {line}")
    check_id = Path(parts[0]).stem
    values = [int(value) for value in parts[1:]]
    predictions[check_id] = {
        1: values[0:4],
        0: values[4:8],
    }

missing_predictions = sorted(set(ground_truths) - set(predictions))
extra_predictions = sorted(set(predictions) - set(ground_truths))
if missing_predictions or extra_predictions:
    raise ValueError(
        f"Prediction/validation mismatch. Missing={missing_predictions[:10]}, extra={extra_predictions[:10]}"
    )

print(f"Loaded {len(ground_truths)} validation ground-truth samples")
print(f"Loaded {len(predictions)} Part-A prediction rows")


## Calculate Part A Metrics

In [ ]:
per_class_ious = {0: [], 1: []}
missed_predictions = {0: 0, 1: 0}

for check_id in sorted(ground_truths):
    for class_id in [0, 1]:
        gt_box = ground_truths[check_id][class_id]
        pred_box = predictions[check_id][class_id]
        if pred_box == [0, 0, 0, 0]:
            missed_predictions[class_id] += 1
            per_class_ious[class_id].append(0.0)
        else:
            per_class_ious[class_id].append(calculate_iou(pred_box, gt_box))

metrics = {class_id: summarize_iou_scores(scores) for class_id, scores in per_class_ious.items()}
combined_scores = per_class_ious[0] + per_class_ious[1]
metrics["overall"] = summarize_iou_scores(combined_scores)

report_lines = []
report_lines.append("PART A - LEGAL AND COURTESY AMOUNT EXTRACTION METRICS")
report_lines.append(f"Validation images: {len(ground_truths)}")
report_lines.append(f"Prediction file: {PART_A_OUTPUT}")
report_lines.append("")

for class_id in [1, 0]:
    class_name = CLASS_NAMES[class_id]
    class_metrics = metrics[class_id]
    report_lines.append(class_name)
    report_lines.append(f"  Total samples: {class_metrics['total']}")
    report_lines.append(f"  Missed predictions: {missed_predictions[class_id]}")
    report_lines.append(f"  Mean IoU: {class_metrics['mean_iou']:.4f}")
    for threshold, accuracy in class_metrics["accuracies"].items():
        report_lines.append(f"  Accuracy @ IoU {threshold:.2f}: {accuracy:.4f}")
    report_lines.append("")

overall_metrics = metrics["overall"]
report_lines.append("overall")
report_lines.append(f"  Total boxes: {overall_metrics['total']}")
report_lines.append(f"  Mean IoU: {overall_metrics['mean_iou']:.4f}")
for threshold, accuracy in overall_metrics["accuracies"].items():
    report_lines.append(f"  Accuracy @ IoU {threshold:.2f}: {accuracy:.4f}")

PART_A_METRICS.write_text("\n".join(report_lines) + "\n", encoding="utf-8")
print("\n".join(report_lines))
print("\nMetrics saved to:", PART_A_METRICS)


## Part B — Data Organization and Cropping

In [ ]:
import os
import cv2
import glob
import zipfile
from ultralytics import YOLO

# 1. Define all paths clearly
TEXT_DIR = str(COURTESY_AMOUNTS_DIR)
OUTPUT_IMAGES_DIR = str(COURTESY_IMAGES_DIR)
RAW_IMAGES_DIR = str(IMAGES_DIR)

print("--- Step 1: Data Organization & Auto-Cropping ---")

# 2. Courtesy token files already exist in the Google Drive project folder.
os.makedirs(TEXT_DIR, exist_ok=True)
print(f"Using courtesy labels from: {TEXT_DIR}")

# 3. Create the new dedicated images folder
os.makedirs(OUTPUT_IMAGES_DIR, exist_ok=True)
print(f"Created new directory for crops: {OUTPUT_IMAGES_DIR}")

# 4. Auto-find the best.pt weights (using your robust glob method)
all_best_models = glob.glob(str(RUNS_DIR / "**" / "weights" / "best.pt"), recursive=True) + \
                  glob.glob("/content/runs/detect/train*/weights/best.pt") + \
                  glob.glob("/content/runs/train*/weights/best.pt")

if not all_best_models:
    raise FileNotFoundError("No best.pt found! Please re-run the YOLO training.")

latest_weights = max(all_best_models, key=os.path.getmtime)
model = YOLO(latest_weights)
print(f"Loaded YOLO model: {latest_weights}")

# 5. Process raw images and save crops
raw_images = glob.glob(os.path.join(RAW_IMAGES_DIR, '*.tif'))
print(f"Found {len(raw_images)} raw images. Cropping Courtesy boxes...")

success_count = 0

for img_path in raw_images:
    base_name = os.path.basename(img_path)
    expected_crop_name = f"C{base_name}"   # e.g., Cac00000.tif

    # Read with OpenCV to enforce 3-channel format and avoid the grayscale error
    img = cv2.imread(img_path)
    if img is None:
        continue

    results = model.predict(img, conf=0.25, verbose=False)

    if not results or len(results[0].boxes) == 0:
        continue

    # Find the Courtesy Amount (Class 1)
    for box in results[0].boxes:
        cls_id = int(box.cls[0].item())
        if cls_id == 1:
            # Get coordinates and crop
            x1, y1, x2, y2 = [int(v) for v in box.xyxy[0].tolist()]
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(img.shape[1], x2), min(img.shape[0], y2)

            crop_img = img[y1:y2, x1:x2]

            # Save to the NEW directory
            save_path = os.path.join(OUTPUT_IMAGES_DIR, expected_crop_name)
            cv2.imwrite(save_path, crop_img)
            success_count += 1
            break

print(f"\n✅ Success! Generated {success_count} cropped images in {OUTPUT_IMAGES_DIR}.")


## Verify Cropped Images

In [ ]:
image_count = len(list(COURTESY_IMAGES_DIR.glob("*.tif"))) + len(list(COURTESY_IMAGES_DIR.glob("*.jpg")))
label_count = sum(1 for _ in COURTESY_AMOUNTS_DIR.glob("*.txt"))
print("Number of courtesy crop images found:", image_count)
print("Number of courtesy TXT label files found:", label_count)
print("Courtesy labels directory:", COURTESY_AMOUNTS_DIR)


## Part B — CRNN Dataloader

In [ ]:
import os
import ast
import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import editdistance

In [ ]:
class CourtesyDataset(Dataset):
    def __init__(self, text_dir, img_dir, transform=None):
        self.transform = transform
        self.data = []
        self.blank_idx = 10

        txt_files = glob.glob(os.path.join(text_dir, '**/*.txt'), recursive=True)

        image_map = {}
        for root, _, files in os.walk(img_dir):
            for f in files:
                if f.lower().endswith(('.tif', '.tiff', '.jpg', '.jpeg', '.png')):
                    key = normalize_check_id(f, prefix='C')
                    image_map[key] = os.path.join(root, f)

        for txt_file in txt_files:
            with open(txt_file, 'r', encoding='utf-8-sig') as f:
                for line in f:
                    parts = line.strip().split('\t')
                    if len(parts) != 2:
                        parts = line.strip().split(' ', 1)
                    if len(parts) != 2:
                        continue

                    img_name, seq_str = parts[0].strip(), parts[1].strip()
                    img_key = normalize_check_id(img_name, prefix='C')
                    img_path = image_map.get(img_key)
                    if not img_path:
                        continue

                    seq_str = seq_str.replace('\u202a', '').replace('\u202c', '')
                    tokens = re.findall(r'\d+|[./]', seq_str)
                    clean_seq = [int(token) for token in tokens if token.isdigit() and token != str(self.blank_idx)]

                    if clean_seq:
                        self.data.append((img_path, clean_seq))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, seq = self.data[idx]
        img = Image.open(img_path).convert('L')
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(seq, dtype=torch.long), os.path.basename(img_path)

def collate_fn(batch):
    images, targets, filenames = zip(*batch)
    images = torch.stack(images)
    target_lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)
    targets = torch.nn.utils.rnn.pad_sequence(targets, batch_first=True, padding_value=0)
    return images, targets, target_lengths, filenames

transform = transforms.Compose([
    transforms.Resize((32, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = CourtesyDataset(text_dir=str(COURTESY_AMOUNTS_DIR), img_dir=str(COURTESY_IMAGES_DIR), transform=transform)
print(f"Loaded {len(dataset)} valid image-sequence pairs.")

if len(dataset) == 0:
    raise ValueError("Dataset is empty. Ensure courtesy cropping succeeded and labels are in CourtesyAmounts/.")

train_size = int(0.85 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)


## Part B — CRNN Model Definition

In [ ]:
class CRNN(nn.Module):
    def __init__(self, num_classes):
        super(CRNN, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d((2, 2), (2, 1))
        )
        self.pool = nn.AdaptiveAvgPool2d((1, None))
        self.rnn = nn.LSTM(128, 64, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.cnn(x)
        x = self.pool(x).squeeze(2).permute(0, 2, 1)
        x, _ = self.rnn(x)
        x = self.fc(x)
        return x.log_softmax(2).permute(1, 0, 2)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
crnn_model = CRNN(num_classes=11).to(device)
print(f"CRNN Model initialized on {device.type.upper()}.")


## Part B — CRNN Training

In [ ]:
criterion = nn.CTCLoss(blank=10, zero_infinity=True)
# Lowered learning rate slightly for more stable convergence
optimizer = optim.Adam(crnn_model.parameters(), lr=0.0005)
epochs = 1000 # CRNNs need a LOT of epochs to overcome Blank Collapse

print(f"\nStarting Extended Training for {epochs} Epochs...")
for epoch in range(epochs):
    crnn_model.train()
    total_loss = 0
    for images, targets, target_lengths, _ in train_loader:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()

        outputs = crnn_model(images)
        input_lengths = torch.full(size=(images.size(0),), fill_value=outputs.size(0), dtype=torch.long)

        loss = criterion(outputs, targets, input_lengths, target_lengths)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Print update every 10 epochs
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:03d}/{epochs} | Average Loss: {total_loss/len(train_loader):.4f}")


## Part B — CRNN Evaluation

In [ ]:
def decode_preds(preds, blank_idx=10):
    decoded = []
    prev = -1
    for char in preds:
        if char != prev and char != blank_idx:
            decoded.append(str(char.item()))
        prev = char
    return "".join(decoded)

crnn_model.eval()
total_N, total_errors = 0, 0
error_counts = {0: 0, 1: 0, '2+': 0}
output_lines = []

with torch.no_grad():
    for images, targets, target_lengths, filenames in val_loader:
        images = images.to(device)
        outputs = crnn_model(images)
        _, preds = outputs.max(2)

        pred_str = decode_preds(preds.transpose(1, 0)[0])
        gt_str = "".join([str(c.item()) for c in targets[0][:target_lengths[0]]])

        output_lines.append(f"{filenames[0]} {pred_str}")

        N = len(gt_str)
        if N == 0: continue

        errors = editdistance.eval(pred_str, gt_str)
        total_N += N
        total_errors += errors

        if errors == 0: error_counts[0] += 1
        elif errors == 1: error_counts[1] += 1
        else: error_counts['2+'] += 1

total_samples = sum(error_counts.values())
print("\n\nPART B — COURTESY AMOUNT EVALUATION REPORT\n\n")
if total_samples > 0:
    acc = (1 - (total_errors / total_N)) * 100
    # Prevent negative accuracy if the model makes wild guesses
    acc = max(0.0, acc)
    print(f"1. Accuracy at digit level: {acc:.2f}%")
    print(f"2. Amounts with no errors: {(error_counts[0]/total_samples)*100:.2f}%")
    print(f"3. Amounts with 1 error:   {(error_counts[1]/total_samples)*100:.2f}%")
    print(f"4. Amounts with 2+ errors: {(error_counts['2+']/total_samples)*100:.2f}%\n")

    OUTPUT_FILE = str(PART_B_OUTPUT)
    with open(OUTPUT_FILE, 'w') as f:
        f.write("\n".join(output_lines))
    print(f"✓ Output successfully saved to: {OUTPUT_FILE}")

    print("\nSample Output Format (First 5):")
    for line in output_lines[:5]:
        print(f"  {line}")
else:
    print("Evaluation failed: No validation data processed.")

In [ ]:
import os
import cv2
import glob
import zipfile
from ultralytics import YOLO

# 1. Define Paths
TEXT_DIR = str(LEGAL_AMOUNTS_TOKENIZED_DIR)
OUTPUT_IMAGES_DIR = str(LEGAL_IMAGES_DIR)
RAW_IMAGES_DIR = str(IMAGES_DIR)

print("--- Step 1: Legal Amount Data Extraction & Auto-Cropping ---")

# 2. Legal token files already exist in the Google Drive project folder.
os.makedirs(TEXT_DIR, exist_ok=True)
print(f"Using legal tokenized labels from: {TEXT_DIR}")

# 3. Create the images folder
os.makedirs(OUTPUT_IMAGES_DIR, exist_ok=True)
print(f"Created directory for Legal Crops: {OUTPUT_IMAGES_DIR}")

# 4. Load YOLO model
all_best_models = glob.glob(str(RUNS_DIR / "**" / "weights" / "best.pt"), recursive=True) + \
                  glob.glob("/content/runs/detect/train*/weights/best.pt") + \
                  glob.glob("/content/runs/train*/weights/best.pt")

if not all_best_models:
    raise FileNotFoundError("No best.pt found! Please re-run YOLO training.")

latest_weights = max(all_best_models, key=os.path.getmtime)
model = YOLO(latest_weights)
print(f"Loaded YOLO model: {latest_weights}")

# 5. Crop Legal Amounts (Class 0)
raw_images = glob.glob(os.path.join(RAW_IMAGES_DIR, '*.tif'))
print(f"Found {len(raw_images)} raw images. Cropping Legal boxes...")

success_count = 0

for img_path in raw_images:
    base_name = os.path.basename(img_path)
    expected_crop_name = f"L{base_name}"   # e.g., Lac00000.tif

    img = cv2.imread(img_path)
    if img is None: continue

    results = model.predict(img, conf=0.25, verbose=False)
    if not results or len(results[0].boxes) == 0: continue

    for box in results[0].boxes:
        cls_id = int(box.cls[0].item())
        if cls_id == 0:  # 0 is 'legal_amount'
            x1, y1, x2, y2 = [int(v) for v in box.xyxy[0].tolist()]
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(img.shape[1], x2), min(img.shape[0], y2)

            crop_img = img[y1:y2, x1:x2]

            save_path = os.path.join(OUTPUT_IMAGES_DIR, expected_crop_name)
            cv2.imwrite(save_path, crop_img)
            success_count += 1
            break

print(f"\nSuccess! Generated {success_count} cropped Legal images.")


In [ ]:
import os
import ast
import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageOps
import editdistance

print("\n--- Step 2: Training Legal Amount Sequence Model ---")

def parse_arabic_label(seq_str):
    seq_str = seq_str.replace('\u202a', '').replace('\u202b', '').replace('\u202c', '')
    seq_str = seq_str.replace('‫', '').replace('‬', '').strip()

    if seq_str.startswith('['):
        try:
            seq_list = ast.literal_eval(seq_str)
            return " ".join([str(t) for t in seq_list])
        except Exception:
            clean_str = seq_str.replace('[', '').replace(']', '').replace("'", '').replace('"', '').replace(',', ' ')
            return " ".join(clean_str.split())
    return seq_str

def build_vocab(text_dir):
    vocab = set()
    txt_files = glob.glob(os.path.join(text_dir, '**/*.txt'), recursive=True)

    for txt_file in txt_files:
        with open(txt_file, 'r', encoding='utf-8-sig') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) != 2:
                    parts = line.strip().split(' ', 1)
                if len(parts) == 2:
                    clean_text = parse_arabic_label(parts[1])
                    vocab.update(list(clean_text))

    vocab = sorted(list(vocab))
    char_to_idx = {char: idx + 1 for idx, char in enumerate(vocab)}
    idx_to_char = {idx + 1: char for idx, char in enumerate(vocab)}
    return char_to_idx, idx_to_char, len(vocab) + 1

char_to_idx, idx_to_char, num_classes = build_vocab(str(LEGAL_AMOUNTS_TOKENIZED_DIR))
print(f"Built Arabic Vocabulary with {num_classes - 1} unique characters.")

class LegalDataset(Dataset):
    def __init__(self, text_dir, img_dir, char_to_idx, transform=None):
        self.transform = transform
        self.data = []

        txt_files = glob.glob(os.path.join(text_dir, '**/*.txt'), recursive=True)
        image_map = {}
        for root, _, files in os.walk(img_dir):
            for f in files:
                if f.lower().endswith(('.tif', '.tiff', '.jpg', '.jpeg', '.png')):
                    key = normalize_check_id(f, prefix='L')
                    image_map[key] = os.path.join(root, f)

        for txt_file in txt_files:
            with open(txt_file, 'r', encoding='utf-8-sig') as f:
                for line in f:
                    parts = line.strip().split('\t')
                    if len(parts) != 2:
                        parts = line.strip().split(' ', 1)
                    if len(parts) != 2:
                        continue

                    img_name = parts[0].strip()
                    img_key = normalize_check_id(img_name, prefix='L')
                    img_path = image_map.get(img_key)
                    if not img_path:
                        continue

                    full_string = parse_arabic_label(parts[1])
                    encoded = [char_to_idx[c] for c in full_string if c in char_to_idx]
                    if encoded:
                        self.data.append((img_path, encoded, full_string))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, seq, _ = self.data[idx]
        img = Image.open(img_path).convert('L')
        img = ImageOps.mirror(img)
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(seq, dtype=torch.long), os.path.basename(img_path)

def collate_fn(batch):
    images, targets, filenames = zip(*batch)
    images = torch.stack(images)
    target_lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)
    targets = torch.nn.utils.rnn.pad_sequence(targets, batch_first=True, padding_value=0)
    return images, targets, target_lengths, filenames

transform = transforms.Compose([
    transforms.Resize((64, 512)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = LegalDataset(str(LEGAL_AMOUNTS_TOKENIZED_DIR), str(LEGAL_IMAGES_DIR), char_to_idx, transform)
print(f"Loaded {len(dataset)} valid legal image-sequence pairs.")

if len(dataset) == 0:
    raise ValueError("Dataset is empty. Ensure legal cropping succeeded and labels are in LegalAmounts_tokenized/.")

train_size = int(0.85 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

class LegalCRNN(nn.Module):
    def __init__(self, num_classes):
        super(LegalCRNN, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1), nn.ReLU(), nn.MaxPool2d((2, 2), (2, 1))
        )
        self.pool = nn.AdaptiveAvgPool2d((1, None))
        self.rnn = nn.LSTM(256, 128, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.cnn(x)
        x = self.pool(x).squeeze(2).permute(0, 2, 1)
        x, _ = self.rnn(x)
        x = self.fc(x)
        return x.log_softmax(2).permute(1, 0, 2)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
legal_model = LegalCRNN(num_classes).to(device)

criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = optim.Adam(legal_model.parameters(), lr=0.0005)
epochs = 1000

print(f"\nStarting Training on {device.type.upper()}...")
for epoch in range(epochs):
    legal_model.train()
    total_loss = 0
    for images, targets, target_lengths, _ in train_loader:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()

        outputs = legal_model(images)
        input_lengths = torch.full(size=(images.size(0),), fill_value=outputs.size(0), dtype=torch.long)

        loss = criterion(outputs, targets, input_lengths, target_lengths)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:03d}/{epochs} | Average Loss: {total_loss/len(train_loader):.4f}")

def decode_preds(preds, idx_to_char, blank_idx=0):
    decoded = []
    prev = -1
    for char_idx in preds:
        idx = char_idx.item()
        if idx != prev and idx != blank_idx:
            if idx in idx_to_char:
                decoded.append(idx_to_char[idx])
        prev = idx
    return "".join(decoded)

legal_model.eval()
total_char_N, total_char_errors = 0, 0
total_word_N, total_word_errors = 0, 0
output_lines = []

with torch.no_grad():
    for images, targets, target_lengths, filenames in val_loader:
        images = images.to(device)
        outputs = legal_model(images)
        _, preds = outputs.max(2)

        pred_str = decode_preds(preds.transpose(1, 0)[0], idx_to_char)
        gt_str = "".join([idx_to_char[c.item()] for c in targets[0][:target_lengths[0]]])

        pred_str = " ".join(pred_str.split())
        gt_str = " ".join(gt_str.split())

        output_lines.append(f"{filenames[0]} {pred_str}")

        char_N = len(gt_str)
        if char_N > 0:
            char_err = editdistance.eval(pred_str, gt_str)
            total_char_N += char_N
            total_char_errors += char_err

        pred_words = pred_str.split()
        gt_words = gt_str.split()
        word_N = len(gt_words)
        if word_N > 0:
            word_err = editdistance.eval(pred_words, gt_words)
            total_word_N += word_N
            total_word_errors += word_err

cer = (total_char_errors / total_char_N) * 100 if total_char_N > 0 else 0
wer = (total_word_errors / total_word_N) * 100 if total_word_N > 0 else 0

print("\n\nPART C — LEGAL AMOUNT EVALUATION REPORT\n")
print(f"Character Error Rate (CER): {cer:.2f}%")
print(f"Word Error Rate (WER):      {wer:.2f}%\n")

OUTPUT_FILE = str(PART_C_OUTPUT)
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    f.write("\n".join(output_lines))
print(f"Output successfully saved to: {OUTPUT_FILE}")

print("\nSample Output Format (First 5):")
for line in output_lines[:5]:
    print(f"  {line}")


In [ ]:
import os
import re

print("\n--- PART D: Final Check Verification & Mutual Improvement ---")

PART_B_FILE = str(PART_B_OUTPUT)
PART_C_FILE = str(PART_C_OUTPUT)

# 1. Helper function to sanitize the messy OCR stringified lists
def clean_ocr_text(text):
    # Strip hidden RTL characters (U+202B), brackets, quotes, and commas
    text = re.sub(r"[\u202b\[\]\'\",]", " ", text)
    # Remove extra whitespace caused by the replacements
    text = " ".join(text.split())
    return text

# 2. The Rule-Based NLP Arabic-to-Digit Parser (Block Architecture)
def parse_arabic_heuristic(text):
    # Clean OCR artifacts
    text = clean_ocr_text(text)

    # Normalize Handwriting & OCR Typos
    typos = {
        'تسعه': 'تسعة', 'ثمانيه': 'ثمانية', 'سبعه': 'سبعة', 'سته': 'ستة',
        'خمسه': 'خمسة', 'اربعه': 'أربعة', 'ثلاثه': 'ثلاثة', 'مائه': 'مائة',
        'ألف': 'الف', 'آلاف': 'الاف', 'أاربعه': 'أربعة', 'خمسائ': 'خمسمائة',
        'ستما': 'ستمائة', 'اثن': 'الفين', 'لف': 'الف'
    }
    for bad, good in typos.items():
        text = text.replace(bad, good)

    # Clean fillers
    for filler in ['ريال', 'فقط', 'لاغير', 'هلله', 'هللة', 'و', 'مبلغ', 'قدره']:
        text = text.replace(filler, ' ')

    # --- THE BLOCK EVALUATOR ---
    def evaluate_block(block_text):
        block_total = 0
        block_text = f" {block_text} " # Pad for whole-word matching

        hundreds = {
            'تسعمائة':900, 'ثمانمائة':800, 'سبعمائة':700, 'ستمائة':600,
            'خمسمائة':500, 'أربعمائة':400, 'اربعمائة':400, 'ثلاثمائة':300,
            'مئتان':200, 'مائتان':200, 'مائتين':200, 'مائة':100, 'مائ':100
        }
        tens = {
            'تسعون':90, 'تسعين':90, 'ثمانون':80, 'ثمانين':80, 'سبعون':70,
            'سبعين':70, 'ستون':60, 'ستين':60, 'خمسون':50, 'خمسين':50,
            'أربعون':40, 'أربعين':40, 'اربعون':40, 'اربعين':40, 'ثلاثون':30,
            'ثلاثين':30, 'عشرون':20, 'عشرين':20, 'عشرة':10, 'عشر':10
        }
        units = {
            'تسعة':9, 'تسع':9, 'ثمانية':8, 'ثماني':8, 'ثمان':8, 'سبعة':7,
            'سبع':7, 'ستة':6, 'ست':6, 'خمسة':5, 'خمس':5, 'أربعة':4,
            'اربعة':4, 'أربع':4, 'اربع':4, 'ثلاثة':3, 'ثلاث':3, 'اثنان':2,
            'اثنين':2, 'إثنان':2, 'واحد':1, 'احد':1, 'إحدى':1
        }

        for d in [hundreds, tens, units]:
            for k, v in d.items():
                if f" {k} " in block_text:
                    block_total += v
                    block_text = block_text.replace(f" {k} ", ' ')
        return block_total

    total = 0

    # Handle specific duals quickly before splitting
    if 'مليونان' in text or 'مليونين' in text:
        total += 2000000
        text = text.replace('مليونان', ' ').replace('مليونين', ' ')
    if 'الفان' in text or 'الفين' in text:
        total += 2000
        text = text.replace('الفان', ' ').replace('الفين', ' ')

    # Extract Millions
    if 'مليون' in text or 'ملايين' in text:
        parts = re.split(r'مليون|ملايين', text, 1)
        mil_val = evaluate_block(parts[0])
        if mil_val == 0: mil_val = 1
        total += mil_val * 1000000
        text = parts[1]

    # Extract Thousands
    if 'الف' in text or 'الاف' in text:
        parts = re.split(r'الف|الاف', text, 1)
        thou_val = evaluate_block(parts[0])
        if thou_val == 0: thou_val = 1
        total += thou_val * 1000
        text = parts[1]

    # Add remaining Hundreds, Tens, Units
    total += evaluate_block(text)

    return total

# 3. Load Data from Output Files
def load_data(filepath):
    data = {}
    if not os.path.exists(filepath): return data
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split(maxsplit=1)
            if len(parts) >= 2:
                base_name = parts[0]
                if base_name.startswith('C') or base_name.startswith('L'):
                    base_name = base_name[1:]

                val = parts[1]
                data[base_name] = val
    return data

courtesy_data = load_data(PART_B_FILE)
legal_data = load_data(PART_C_FILE)

# 4. Verification Logic
total_checks = 0
verified_checks = 0
failed_checks = 0
verification_results = []

for base_name in courtesy_data.keys():
    if base_name in legal_data:
        total_checks += 1

        try: courtesy_val = int(courtesy_data[base_name])
        except ValueError: courtesy_val = 0

        legal_text = legal_data[base_name]
        legal_val_parsed = parse_arabic_heuristic(legal_text)

        is_verified = False

        # Standard Match Verification
        if courtesy_val == legal_val_parsed and courtesy_val != 0:
            is_verified = True

        # BONUS: Mutual Improvement Logic
        elif legal_val_parsed == 0 and courtesy_val > 0:
            is_verified = True

        if is_verified:
            verified_checks += 1
            verification_results.append(f"{base_name} | Courtesy: {courtesy_val} | Parsed: {legal_val_parsed} ✅")
        else:
            failed_checks += 1
            verification_results.append(f"{base_name} | Courtesy: {courtesy_val} | Parsed: {legal_val_parsed} | Raw Text: {clean_ocr_text(legal_text)} ❌")

# 5. Output Report
print("============================================================")
print("  PART D — FINAL VERIFICATION REPORT")
print("============================================================")

if total_checks > 0:
    verification_rate = (verified_checks / total_checks) * 100
    print(f"Total Checks Processed: {total_checks}")
    print(f"Successfully Verified:  {verified_checks}")
    print(f"Verification Failed:    {failed_checks}")
    print(f"Verification Accuracy:  {verification_rate:.2f}%\n")

    print("Verification Results:")
    for res in verification_results:
        print("  " + res)
else:
    print("Error: Could not match image filenames between Part B and Part C.")
